Problem: Choosing three restaurants during holiday (120 days)

- Italian restaurant: avg score 8/10 
- Chinese restaurant: avg score 6/10
- Mexican restaurant: avg score 9/10 


** Exploration: not sure which restaurant is the best, so try different ones (waste time trying bad restaurants)

** Exploitation: know which restaurants seems best, so keep going there (might miss a better one)

Therefore, it needs MAB to balance the two.



- Agent = decision maker
- Restaurant = arms (one possible action can be chosen)
- Score = reward
- Goal = maximise total satisfaction



Learning: 

Epsilon-Greedy algorithm is a simple method for managing exploration and exploitation:

- prob epsilon: explore (random arm) 
- prob 1-epsilon: exploit (best estimated arm)

Limit: 

- Exploration is random
- Explore even bad arms 
- It does not reduce exploration intelligently

In [1]:
import numpy as np

In [4]:
class EpsilonGreedy: 
    def __init__(self, n_restaurants, epsilon): 
        self.n_restaurants = n_restaurants
        self.epsilon = epsilon
        self.visits = np.zeros(n_restaurants)
        self.satisfaction = np.zeros(n_restaurants) # the total reward accumulated for each restaurant
    
    def choose_restaurant(self):
        if np.random.random() < self.epsilon:
            return np.random.choice(self.n_restaurants) # exploration
        else: 
            return np.argmax(self.satisfaction / (self.visits + 1e-5)) # exploitation

    def update(self, restaurant, score):
        self.visits[restaurant] += 1
        self.satisfaction[restaurant] += score

n_restaurants = 3
epsilon = 0.1 
n_days = 120

true_avg_satisfaction = np.array([8,6,9])
true_stddev_satisfaction = np.array([1,2,1.5])

total_satisfaction_arr = []
for i in range(50): # run the simulation 50 times 
    epsilon_greedy_restaurant = EpsilonGreedy(n_restaurants, epsilon)
    total_satisfaction = 0 

    for _ in range(n_days): 
        restaurant = epsilon_greedy_restaurant.choose_restaurant()
        score = np.random.normal(loc=true_avg_satisfaction[restaurant], scale = true_stddev_satisfaction[restaurant])
        epsilon_greedy_restaurant.update(restaurant, score)
        total_satisfaction += score
    
    print("Total Satisfaction (Epsilon-Greedy):", total_satisfaction)
    total_satisfaction_arr.append(total_satisfaction)

# Calculate average satisfaction
np.mean(total_satisfaction_arr) / n_days, np.std(total_satisfaction_arr) / n_days


Total Satisfaction (Epsilon-Greedy): 1041.0606294253425
Total Satisfaction (Epsilon-Greedy): 1003.3341032957364
Total Satisfaction (Epsilon-Greedy): 990.0956393758901
Total Satisfaction (Epsilon-Greedy): 961.1516104257108
Total Satisfaction (Epsilon-Greedy): 934.4128967389621
Total Satisfaction (Epsilon-Greedy): 1041.1232858149517
Total Satisfaction (Epsilon-Greedy): 1030.7741135574133
Total Satisfaction (Epsilon-Greedy): 1007.6361812523944
Total Satisfaction (Epsilon-Greedy): 1032.8760517138148
Total Satisfaction (Epsilon-Greedy): 1052.2301249213403
Total Satisfaction (Epsilon-Greedy): 1034.2160113022626
Total Satisfaction (Epsilon-Greedy): 937.5857687541225
Total Satisfaction (Epsilon-Greedy): 1077.6770171432388
Total Satisfaction (Epsilon-Greedy): 1060.0278136736301
Total Satisfaction (Epsilon-Greedy): 1018.9908602275098
Total Satisfaction (Epsilon-Greedy): 1033.6944336295837
Total Satisfaction (Epsilon-Greedy): 1044.8987993323933
Total Satisfaction (Epsilon-Greedy): 1039.5406504209

(np.float64(8.462421680010431), np.float64(0.3334745662149465))

Upper Confidence Bound

- Balance exploration and exploitation: restaurants that haven't been visited enough are explored to reduce uncertainty
  
$$
A_t = \arg\max_i \left( \bar{X}_i + c \sqrt{\frac{\ln t}{n_i}} \right)
$$

- The first part: exploitation -> the everage satisfaction score of restaurant i 
- The second part: exploration bonus
  - If the restaurant was visited very few times -> $n_i$ is small and denominator small -> term is big
  - If the restaurant was visited many times -> $n_i$ is large -> denominator large -> term is small

In [7]:
class UCB: 
    def __init__(self, n_restaurants):
        self.n_restaurants = n_restaurants
        self.visits = np.zeros(n_restaurants)
        self.satisfaction = np.zeros(n_restaurants)
        self.total_trials = 0 

    def choose_restaurant(self): 
        if self.total_trials < self.n_restaurants: 
            return self.total_trials # must visit restaurant at least once
        
        ucb_values = np.zeros(self.n_restaurants)
        for restaurant in range(self.n_restaurants):
            avg_score = self.satisfaction[restaurant] / (self.visits[restaurant] + 1e-5)
            confidence_bound = np.sqrt(2 * np.log(self.total_trials + 1) / (self.visits[restaurant] + 1e-5))
            ucb_values[restaurant] = avg_score + confidence_bound
        return np.argmax(ucb_values)
    
    def update(self, restaurant, score):
        self.visits[restaurant] += 1
        self.satisfaction[restaurant] += score 
        self.total_trials +=1 


total_satisfaction_arr_ucb = []
for i in range(50):  # Run the simulation 50 times
    ucb_restaurant = UCB(n_restaurants)
    total_satisfaction = 0

    for _ in range(n_days):
        restaurant = ucb_restaurant.choose_restaurant()
        score = np.random.normal(loc=true_avg_satisfaction[restaurant], scale=true_stddev_satisfaction[restaurant])
        ucb_restaurant.update(restaurant, score)
        total_satisfaction += score

    print("Total Satisfaction (UCB):", total_satisfaction)
    total_satisfaction_arr_ucb.append(total_satisfaction)

# Calculate average satisfaction
np.mean(total_satisfaction_arr_ucb) / n_days, np.std(total_satisfaction_arr_ucb) / n_days

Total Satisfaction (UCB): 1057.5775866642653
Total Satisfaction (UCB): 1064.8849302504861
Total Satisfaction (UCB): 1054.0927029258319
Total Satisfaction (UCB): 1057.4613196779626
Total Satisfaction (UCB): 1055.239352895858
Total Satisfaction (UCB): 1048.4104616859458
Total Satisfaction (UCB): 1065.8428322779446
Total Satisfaction (UCB): 1044.5133896223663
Total Satisfaction (UCB): 1038.1974002714062
Total Satisfaction (UCB): 1062.600590692574
Total Satisfaction (UCB): 1065.9282062682264
Total Satisfaction (UCB): 1086.6829830901263
Total Satisfaction (UCB): 1073.367883220372
Total Satisfaction (UCB): 1043.2588524287128
Total Satisfaction (UCB): 1073.2913942540026
Total Satisfaction (UCB): 1075.8390127950029
Total Satisfaction (UCB): 1091.5652454270112
Total Satisfaction (UCB): 1068.3883594390607
Total Satisfaction (UCB): 1068.9181107682214
Total Satisfaction (UCB): 1090.7909759312283
Total Satisfaction (UCB): 1079.1224371192511
Total Satisfaction (UCB): 1075.4933958503539
Total Satisfa

(np.float64(8.875438823153768), np.float64(0.16829827834903285))

Thompson's Sampling 

- Use probability to balance exploration and exploitation
- Example: 
  - Restaurant A: avg. ~4.5
  - Restaurant B: avg. ~4.3
  
  -> Thompson: for each restaurant, sample a possible true rating from its distribution
  
  -> Choose the restaurant with highest sampled value
- Not require hyperparameter tuning (compared to UCB - hard to tune)

In [8]:
class ThompsonSampling:
    def __init__(self, n_restaurants):
        self.n_restaurants = n_restaurants
        self.visits = np.zeros(n_restaurants)
        self.satisfaction = np.zeros(n_restaurants)
        self.alpha = np.ones(n_restaurants)  # Beta distribution parameters
        self.beta = np.ones(n_restaurants)

    def choose_restaurant(self):
        sampled_values = np.random.beta(self.alpha, self.beta)
        return np.argmax(sampled_values)

    def update(self, restaurant, score):
        self.visits[restaurant] += 1
        self.satisfaction[restaurant] += score
        # Update the beta distribution based on the satisfaction score
        if score > np.mean(self.satisfaction / (self.visits + 1e-5)):
            self.alpha[restaurant] += 1  # success
        else:
            self.beta[restaurant] += 1  # failure

n_restaurants = 3
n_days = 120

true_avg_satisfaction = np.array([8, 6, 9])
true_stddev_satisfaction = np.array([1, 2, 1.5])

total_satisfaction_arr = []
for i in range(50):  # Run the simulation 50 times
    thompson_sampling_restaurant = ThompsonSampling(n_restaurants)
    total_satisfaction = 0

    for _ in range(n_days):
        restaurant = thompson_sampling_restaurant.choose_restaurant()
        score = np.random.normal(loc=true_avg_satisfaction[restaurant], scale=true_stddev_satisfaction[restaurant])
        thompson_sampling_restaurant.update(restaurant, score)
        total_satisfaction += score

    print("Total Satisfaction (Thompson Sampling):", total_satisfaction)
    total_satisfaction_arr.append(total_satisfaction)

# Calculate average satisfaction
np.mean(total_satisfaction_arr) / n_days, np.std(total_satisfaction_arr) / n_days

Total Satisfaction (Thompson Sampling): 1030.0873251142189
Total Satisfaction (Thompson Sampling): 946.9824155854963
Total Satisfaction (Thompson Sampling): 1058.5855334399064
Total Satisfaction (Thompson Sampling): 1053.4534984602176
Total Satisfaction (Thompson Sampling): 1058.0283203915408
Total Satisfaction (Thompson Sampling): 1026.862500614115
Total Satisfaction (Thompson Sampling): 944.5649148786324
Total Satisfaction (Thompson Sampling): 1008.9720894071052
Total Satisfaction (Thompson Sampling): 1014.0401265180953
Total Satisfaction (Thompson Sampling): 1055.8037741020005
Total Satisfaction (Thompson Sampling): 1077.417842096566
Total Satisfaction (Thompson Sampling): 943.1124985816879
Total Satisfaction (Thompson Sampling): 1028.8104584164016
Total Satisfaction (Thompson Sampling): 1032.0528976124956
Total Satisfaction (Thompson Sampling): 972.9161363380165
Total Satisfaction (Thompson Sampling): 1038.621298184489
Total Satisfaction (Thompson Sampling): 991.211715736899
Total 

(np.float64(8.548897828944733), np.float64(0.3283871087035346))